*Don't forget to save a copy of this notebook in your Drive before working with it!*

# Plotly and Bokeh

## A simple data set

In [ ]:
import pandas as pd

In [ ]:
data = pd.read_csv("https://github.com/cbrown-clu/class_data/raw/refs/heads/main/data/iris.csv")

In [ ]:
data.head()

In [ ]:
data.plot(kind="scatter", x="sepal length", y="sepal width")

## With plotly

A simple scatter plot with hover over information.

In [ ]:
import plotly.express as px

fig = px.scatter(data, x="sepal length", y="sepal width", color='species', hover_data=['species', 'petal length'])
fig.show()

Similar plot but visualizing petal length as a bubble radius instead.

In [ ]:
import plotly.express as px

fig = px.scatter(data, x="sepal length", y="sepal width",
                 size="petal width", color="species",
                 hover_data=['species', 'petal length'],
                 opacity=0.4)
fig.show()

And a box plot.

In [ ]:
import plotly.express as px

fig = px.box(data, x="species", y="petal length", color="species")
fig.show()

## With bokeh

In [ ]:
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import HoverTool, ColumnDataSource
from bokeh.palettes import Category10
from bokeh.transform import factor_cmap

output_notebook()

# Create a ColumnDataSource from the pandas DataFrame for better integration with Bokeh tools
source = ColumnDataSource(data)

# Define the tooltips for the hover tool
hover = HoverTool(tooltips=[
    ("Petal Length", "@{petal length}{0.00}"), # Corrected syntax for column names with spaces
    ("Petal Width", "@{petal width}{0.00}"), # Corrected syntax for column names with spaces
    ("Species", "@species")
])

p = figure(title="Petal Length vs Petal Width", x_axis_label='Petal Length', y_axis_label='Petal Width',
           tools=[hover, "pan", "wheel_zoom", "box_zoom", "reset", "save"])

species_list = data['species'].unique().tolist()

p.scatter(x='petal length', y='petal width', source=source,
          legend_field='species',
          color=factor_cmap('species', palette=Category10[len(species_list)], factors=species_list),
          size=8)

show(p)

## A more complex data set

In [ ]:
import pandas as pd
data = pd.read_csv("https://github.com/cbrown-clu/class_data/raw/refs/heads/main/data/user_behavior_dataset.csv")
data.head()

In [ ]:
import plotly.express as px

fig = px.scatter_3d(data, x='Screen On Time (hours/day)',
                    y='Number of Apps Installed',
                    z='Age',
                    color='User Behavior Class',
                    title='3D Scatter Plot of User Behavior')
fig.show()

In [ ]:
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.transform import cumsum
from bokeh.palettes import Category10, Spectral6
from bokeh.models import ColumnDataSource, LabelSet
from bokeh.layouts import row
import pandas as pd
import numpy as np

output_notebook()

def create_pie_chart(df, column_name, title, palette=Category10):
    counts = df[column_name].value_counts()
    data = pd.DataFrame({'value': counts.index, 'count': counts.values})
    data['angle'] = data['count'] / data['count'].sum() * 2 * np.pi

    num_categories = len(data['value'])

    # Determine the base palette to use. Category10 is a dict, get its full list of colors.
    if palette == Category10:
        effective_palette = Category10[10] # Use the full 10 colors as base
    elif palette == Spectral6:
        effective_palette = Spectral6
    else:
        effective_palette = Category10[10] # Default fallback

    # Ensure we have enough colors by repeating the effective_palette if needed
    # and then taking only the required number of colors.
    current_palette = (effective_palette * (num_categories // len(effective_palette) + 1))[:num_categories]

    data['color'] = current_palette

    source = ColumnDataSource(data)

    p = figure(height=350, title=title, toolbar_location=None,
               tools="hover", tooltips="@value: @count", x_range=(-0.5, 1.0))

    p.wedge(x=0, y=1,
            radius=0.4,
            start_angle=cumsum('angle', include_zero=True),
            end_angle=cumsum('angle'),
            line_color="white",
            fill_color='color',
            legend_field='value',
            source=source)

    p.axis.axis_label = None
    p.axis.visible = False
    p.grid.grid_line_color = None

    return p

# Create pie charts for the requested columns
p_os = create_pie_chart(data, 'Operating System', 'Operating System Distribution')
p_gender = create_pie_chart(data, 'Gender', 'Gender Distribution')
p_behavior = create_pie_chart(data, 'User Behavior Class', 'User Behavior Class Distribution', palette=Spectral6)

# Arrange them side by side
show(row(p_os, p_gender, p_behavior))